# Prénoms en France (INSEE, 1900–2020)

Opens `nat2020.pq` (births per first name, sex and year). Runs unchanged **in the browser**
(JupyterLite: the file ships with the site) and **locally** (`pixi run get-data` then `pixi run lab`).

In [ ]:
import hvplot.pandas  # noqa: F401  (registers .hvplot on DataFrames)
import panel as pn

import prenoms

pn.extension()

df = prenoms.load_nat2020("nat2020.pq")
df

## Overview

In [ ]:
df.info()
print(f"{df['preusuel'].nunique():,} distinct first names")
print(f"years {df['annais'].min()}–{df['annais'].max()}")

In [ ]:
births = df.dropna(subset=["annais"]).groupby(["annais", "sexe"])["nombre"].sum().unstack()
births.columns = ["boys", "girls"]
births.hvplot.line(title="Births recorded per year", ylabel="births", height=300, responsive=True)

## Most given names

In [ ]:
named = df[df["preusuel"] != "_PRENOMS_RARES"]
named.groupby(["preusuel", "sexe"])["nombre"].sum().nlargest(20)

## One name over time

In [ ]:
name = pn.widgets.AutocompleteInput(
    name="first name", options=sorted(named["preusuel"].dropna().unique()), value="MARIE"
)


def plot(n):
    d = named[named["preusuel"] == n].dropna(subset=["annais"])
    return d.hvplot.line(
        x="annais", y="nombre", by="sexe", title=n, xlabel="year", height=300, responsive=True
    )


pn.Column(name, pn.bind(plot, name))